In [1]:
# 1) Mount Google Drive, load chunks, and define helper functions
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
from datetime import datetime, timezone
from collections import Counter

drive.mount("/content/drive")

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
GDRIVE_INPUT_PATH = GDRIVE_PROJECT_DIR / "2wikimultihopqa_docs_chunks.json"

KG_DIR = GDRIVE_PROJECT_DIR / "kg" / "2wikimultihopqa"
KG_DIR.mkdir(parents=True, exist_ok=True)

assert GDRIVE_INPUT_PATH.exists(), f"Input file not found: {GDRIVE_INPUT_PATH}"

with open(GDRIVE_INPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

assert isinstance(chunks, list), "The input JSON must be a list of chunks."

def expected_chunk_id(input_index):
    # Return the expected chunk id for a zero-based index.
    return f"2wikimultihopqa_chunk_{input_index + 1:08d}"

def atomic_json_dump(obj, path):
    # Write JSON safely, then atomically replace the target file.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def summarize_ranges(values, max_ranges=30):
    # Convert integer values into compact human-readable ranges.
    values = sorted(set(values))

    if not values:
        return []

    ranges = []
    start = prev = values[0]

    for x in values[1:]:
        if x == prev + 1:
            prev = x
        else:
            ranges.append((start, prev))
            start = prev = x

    ranges.append((start, prev))

    out = [
        str(a) if a == b else f"{a}-{b}"
        for a, b in ranges[:max_ranges]
    ]

    if len(ranges) > max_ranges:
        out.append(f"... plus {len(ranges) - max_ranges} more ranges")

    return out

def input_index_from_item(item):
    # Prefer explicit input_index, otherwise recover it from chunk_id.
    idx = item.get("input_index")

    if isinstance(idx, int):
        return idx

    chunk_id = item.get("chunk_id")

    if isinstance(chunk_id, str):
        import re
        m = re.search(r"2wikimultihopqa_chunk_(\d{8})$", chunk_id)
        if m:
            return int(m.group(1)) - 1

    return None

def is_usable_success(item, idx):
    # HotpotQA-style usable-success check.
    if not isinstance(item, dict):
        return False

    if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
        return False

    if item.get("chunk_id") != expected_chunk_id(idx):
        return False

    if item.get("error") is not None:
        return False

    if item.get("finish_reason") == "length":
        return False

    if not isinstance(item.get("entities"), list):
        return False

    if not isinstance(item.get("relations"), list):
        return False

    if not isinstance(item.get("facts"), list):
        return False

    return True

def failure_reasons_hotpot_style(item, idx):
    # Report only real structural/run failures.
    reasons = []

    if not isinstance(item, dict):
        return ["item_not_dict"]

    if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
        return ["invalid_input_index"]

    if item.get("chunk_id") != expected_chunk_id(idx):
        reasons.append("chunk_id_mismatch")

    if item.get("error") is not None:
        reasons.append("error_not_none")

    if item.get("finish_reason") == "length":
        reasons.append("finish_reason_length")

    if not isinstance(item.get("entities"), list):
        reasons.append("entities_not_list")

    if not isinstance(item.get("relations"), list):
        reasons.append("relations_not_list")

    if not isinstance(item.get("facts"), list):
        reasons.append("facts_not_list")

    return sorted(set(reasons))

TOTAL_CHUNKS = len(chunks)

CLEAN_OUTPUT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean.json"
CLEAN_AUDIT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean_audit.json"
UNRESOLVED_CHUNKS_PATH = KG_DIR / "2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.json"

MANUAL_REPAIR_ITEM_PATH = KG_DIR / "2wikimultihopqa_kg_manual_repair_chunk_00010637.json"
MANUAL_REPAIR_META_PATH = KG_DIR / "2wikimultihopqa_kg_manual_repair_chunk_00010637_meta.json"

CLEAN_BACKUP_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean.before_manual_repair.backup.json"
CLEAN_AUDIT_BACKUP_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean_audit.before_manual_repair.backup.json"
UNRESOLVED_BACKUP_PATH = KG_DIR / "2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.before_manual_repair.backup.json"

CLEAN_MANUAL_REPAIRED_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean.manual_repaired.json"
CLEAN_MANUAL_REPAIRED_AUDIT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{TOTAL_CHUNKS:08d}.clean.manual_repaired_audit.json"

print("Total chunks:", TOTAL_CHUNKS)
print("First chunk:", chunks[0]["Chunk_id"])
print("Last chunk:", chunks[-1]["Chunk_id"])
print("KG_DIR:", KG_DIR)
print("Clean JSON:", CLEAN_OUTPUT_PATH)

assert CLEAN_OUTPUT_PATH.exists(), f"Clean final JSON not found: {CLEAN_OUTPUT_PATH}"

# Validate chunk id order.
chunk_id_mismatches = []

for i, chunk in enumerate(chunks):
    got = chunk.get("Chunk_id")
    expected = expected_chunk_id(i)

    if got != expected:
        chunk_id_mismatches.append({
            "input_index": i,
            "expected_chunk_id": expected,
            "got_chunk_id": got,
        })

if chunk_id_mismatches:
    print(json.dumps(chunk_id_mismatches[:10], ensure_ascii=False, indent=2))
    raise RuntimeError("Chunk_id order does not match input_index order.")

print("Chunk id order is valid.")

Mounted at /content/drive
Total chunks: 12685
First chunk: 2wikimultihopqa_chunk_00000001
Last chunk: 2wikimultihopqa_chunk_00012685
KG_DIR: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa
Clean JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.json
Chunk id order is valid.


In [2]:
# 2) Define the manual repair item for the remaining failed chunk

MANUAL_INPUT_INDEX = 10636
MANUAL_CHUNK_NUMBER = MANUAL_INPUT_INDEX + 1

manual_chunk = chunks[MANUAL_INPUT_INDEX]

assert manual_chunk["Chunk_id"] == "2wikimultihopqa_chunk_00010637"
assert manual_chunk["Title"] == "Youssef Chahine"

manual_kg_payload = {
    "entities": [
        "Youssef Chahine",
        "25 January 1926",
        "27 July 2008",
        "Egyptian film industry",
        "1950",
        "Cannes 50th Anniversary Award",
        "lifetime achievement",
        "Omar Sharif",
        "film festivals",
        "11'9\"01 September 11",
        "2002"
    ],
    "relations": [
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was born on 25 January 1926.",
            "tail": "25 January 1926"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine died on 27 July 2008.",
            "tail": "27 July 2008"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was active in the Egyptian film industry.",
            "tail": "Egyptian film industry"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was active in the Egyptian film industry from 1950.",
            "tail": "1950"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine won the Cannes 50th Anniversary Award.",
            "tail": "Cannes 50th Anniversary Award"
        },
        {
            "head": "Cannes 50th Anniversary Award",
            "relation": "The Cannes 50th Anniversary Award was for lifetime achievement.",
            "tail": "lifetime achievement"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was credited with launching the career of Omar Sharif.",
            "tail": "Omar Sharif"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was often present at film festivals.",
            "tail": "film festivals"
        },
        {
            "head": "Youssef Chahine",
            "relation": "Youssef Chahine was one of the co-directors of 11'9\"01 September 11.",
            "tail": "11'9\"01 September 11"
        },
        {
            "head": "11'9\"01 September 11",
            "relation": "11'9\"01 September 11 is associated with 2002.",
            "tail": "2002"
        }
    ],
    "facts": [
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine was an Egyptian film director."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine was born on 25 January 1926 and died on 27 July 2008."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine was active in the Egyptian film industry from 1950 until his death."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine won the Cannes 50th Anniversary Award for lifetime achievement."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine was credited with launching the career of actor Omar Sharif."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine was often present at film festivals during the earlier decades of his work."
        },
        {
            "entity": "Youssef Chahine",
            "info": "Youssef Chahine gained his largest international audiences as one of the co-directors of 11'9\"01 September 11."
        },
        {
            "entity": "11'9\"01 September 11",
            "info": "11'9\"01 September 11 is a 2002 work mentioned in this chunk."
        },
        {
            "entity": "Omar Sharif",
            "info": "Omar Sharif's career was credited as having been launched by Youssef Chahine."
        },
        {
            "entity": "Cannes 50th Anniversary Award",
            "info": "The Cannes 50th Anniversary Award won by Youssef Chahine was for lifetime achievement."
        }
    ]
}

manual_repair_item = {
    "chunk_id": manual_chunk["Chunk_id"],
    "title": manual_chunk["Title"],
    "paragraph_id": manual_chunk["Paragraph_id"],
    "token_count": manual_chunk.get("Token_count"),
    "entities": manual_kg_payload["entities"],
    "relations": manual_kg_payload["relations"],
    "facts": manual_kg_payload["facts"],
    "error": None,
    "latency_sec": 0.0,
    "finish_reason": "manual_repair",
    "max_tokens_used": None,
    "usage": None,
    "attempts": [],
    "input_index": MANUAL_INPUT_INDEX,
    "manual_repair": True,
    "repair_run": True,
    "repair_pass": "manual_repair",
    "manual_repair_created_at_utc": datetime.now(timezone.utc).isoformat(),
}

print("Manual repair item:")
print(json.dumps(manual_repair_item, ensure_ascii=False, indent=2))

Manual repair item:
{
  "chunk_id": "2wikimultihopqa_chunk_00010637",
  "title": "Youssef Chahine",
  "paragraph_id": [
    1
  ],
  "token_count": 124,
  "entities": [
    "Youssef Chahine",
    "25 January 1926",
    "27 July 2008",
    "Egyptian film industry",
    "1950",
    "Cannes 50th Anniversary Award",
    "lifetime achievement",
    "Omar Sharif",
    "film festivals",
    "11'9\"01 September 11",
    "2002"
  ],
  "relations": [
    {
      "head": "Youssef Chahine",
      "relation": "Youssef Chahine was born on 25 January 1926.",
      "tail": "25 January 1926"
    },
    {
      "head": "Youssef Chahine",
      "relation": "Youssef Chahine died on 27 July 2008.",
      "tail": "27 July 2008"
    },
    {
      "head": "Youssef Chahine",
      "relation": "Youssef Chahine was active in the Egyptian film industry.",
      "tail": "Egyptian film industry"
    },
    {
      "head": "Youssef Chahine",
      "relation": "Youssef Chahine was active in the Egyptian film industr

In [3]:
# 3) Validate the manual repair item

def validate_manual_kg_payload(payload):
    # Validate top-level KG payload constraints.
    errors = []

    if not isinstance(payload, dict):
        return ["payload_not_dict"]

    required_keys = ["entities", "relations", "facts"]

    for key in required_keys:
        if key not in payload:
            errors.append(f"missing_key:{key}")

    entities = payload.get("entities")
    relations = payload.get("relations")
    facts = payload.get("facts")

    if not isinstance(entities, list):
        errors.append("entities_not_list")
        entities = []

    if not isinstance(relations, list):
        errors.append("relations_not_list")
        relations = []

    if not isinstance(facts, list):
        errors.append("facts_not_list")
        facts = []

    if len(entities) > 30:
        errors.append("too_many_entities")

    if len(relations) > 35:
        errors.append("too_many_relations")

    if len(facts) > 30:
        errors.append("too_many_facts")

    entity_set = set()

    for ent in entities:
        if not isinstance(ent, str) or not ent.strip():
            errors.append("invalid_entity_string")
        else:
            entity_set.add(ent)

    if len(entity_set) != len(entities):
        errors.append("duplicate_entities")

    seen_relations = set()

    for rel in relations:
        if not isinstance(rel, dict):
            errors.append("relation_not_dict")
            continue

        allowed_rel_keys = {"head", "relation", "tail"}

        if set(rel.keys()) != allowed_rel_keys:
            errors.append("relation_has_wrong_keys")

        head = rel.get("head")
        tail = rel.get("tail")
        relation_text = rel.get("relation")

        if head not in entity_set:
            errors.append(f"relation_head_not_in_entities:{head}")

        if tail not in entity_set:
            errors.append(f"relation_tail_not_in_entities:{tail}")

        if not isinstance(relation_text, str) or not relation_text.strip():
            errors.append("invalid_relation_text")

        rel_key = (head, relation_text, tail)
        if rel_key in seen_relations:
            errors.append("duplicate_relation")
        seen_relations.add(rel_key)

    seen_facts = set()

    for fact in facts:
        if not isinstance(fact, dict):
            errors.append("fact_not_dict")
            continue

        allowed_fact_keys = {"entity", "info"}

        if set(fact.keys()) != allowed_fact_keys:
            errors.append("fact_has_wrong_keys")

        entity = fact.get("entity")
        info = fact.get("info")

        if entity not in entity_set:
            errors.append(f"fact_entity_not_in_entities:{entity}")

        if not isinstance(info, str) or not info.strip():
            errors.append("invalid_fact_info")

        fact_key = (entity, info)
        if fact_key in seen_facts:
            errors.append("duplicate_fact")
        seen_facts.add(fact_key)

    return sorted(set(errors))

manual_payload_errors = validate_manual_kg_payload(manual_kg_payload)
manual_item_failure_reasons = failure_reasons_hotpot_style(manual_repair_item, MANUAL_INPUT_INDEX)

print("Manual payload strict validation errors:", manual_payload_errors)
print("Manual item hotpot-style failure reasons:", manual_item_failure_reasons)
print("Manual item usable success:", is_usable_success(manual_repair_item, MANUAL_INPUT_INDEX))

if manual_payload_errors:
    raise RuntimeError("Manual KG payload failed strict validation.")

if manual_item_failure_reasons:
    raise RuntimeError("Manual repair item failed usable-success validation.")

atomic_json_dump(manual_repair_item, MANUAL_REPAIR_ITEM_PATH)

manual_meta = {
    "dataset": "2wikimultihopqa",
    "manual_repair_item_path": str(MANUAL_REPAIR_ITEM_PATH),
    "input_index_0_based": MANUAL_INPUT_INDEX,
    "chunk_number_1_based": MANUAL_CHUNK_NUMBER,
    "chunk_id": manual_chunk["Chunk_id"],
    "title": manual_chunk["Title"],
    "payload_validation": "passed",
    "usable_success": True,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(manual_meta, MANUAL_REPAIR_META_PATH)

print("Saved manual repair item:", MANUAL_REPAIR_ITEM_PATH)
print("Saved manual repair meta:", MANUAL_REPAIR_META_PATH)

Manual payload strict validation errors: []
Manual item hotpot-style failure reasons: []
Manual item usable success: True
Saved manual repair item: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_manual_repair_chunk_00010637.json
Saved manual repair meta: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_manual_repair_chunk_00010637_meta.json


In [4]:
# 4) Load the clean final JSON and create backups

with open(CLEAN_OUTPUT_PATH, "r", encoding="utf-8") as f:
    clean_items = json.load(f)

assert isinstance(clean_items, list), "Clean output must be a list."
assert len(clean_items) == TOTAL_CHUNKS, f"Expected {TOTAL_CHUNKS} items, got {len(clean_items)}."

old_item = clean_items[MANUAL_INPUT_INDEX]

print("Old item at manual repair index:")
print("input_index:", MANUAL_INPUT_INDEX)
print("chunk_id:", old_item.get("chunk_id"))
print("title:", old_item.get("title"))
print("error:", old_item.get("error"))
print("finish_reason:", old_item.get("finish_reason"))
print("failure_reasons:", failure_reasons_hotpot_style(old_item, MANUAL_INPUT_INDEX))

assert old_item.get("chunk_id") == manual_chunk["Chunk_id"], "The target clean item has a different chunk_id."

# Create backups only once.
if not CLEAN_BACKUP_PATH.exists():
    shutil.copy2(CLEAN_OUTPUT_PATH, CLEAN_BACKUP_PATH)
    print("Created clean backup:", CLEAN_BACKUP_PATH)
else:
    print("Clean backup already exists:", CLEAN_BACKUP_PATH)

if CLEAN_AUDIT_PATH.exists() and not CLEAN_AUDIT_BACKUP_PATH.exists():
    shutil.copy2(CLEAN_AUDIT_PATH, CLEAN_AUDIT_BACKUP_PATH)
    print("Created clean audit backup:", CLEAN_AUDIT_BACKUP_PATH)
elif CLEAN_AUDIT_BACKUP_PATH.exists():
    print("Clean audit backup already exists:", CLEAN_AUDIT_BACKUP_PATH)

if UNRESOLVED_CHUNKS_PATH.exists() and not UNRESOLVED_BACKUP_PATH.exists():
    shutil.copy2(UNRESOLVED_CHUNKS_PATH, UNRESOLVED_BACKUP_PATH)
    print("Created unresolved backup:", UNRESOLVED_BACKUP_PATH)
elif UNRESOLVED_BACKUP_PATH.exists():
    print("Unresolved backup already exists:", UNRESOLVED_BACKUP_PATH)

Old item at manual repair index:
input_index: 10636
chunk_id: 2wikimultihopqa_chunk_00010637
title: Youssef Chahine
error: finish_reason=length at max_tokens=6144
finish_reason: length
failure_reasons: ['entities_not_list', 'error_not_none', 'facts_not_list', 'finish_reason_length', 'relations_not_list']
Created clean backup: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.before_manual_repair.backup.json
Created clean audit backup: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean_audit.before_manual_repair.backup.json
Created unresolved backup: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.before_manual_repair.backup.json


In [5]:
# 5) Patch the clean final JSON with the manual repair item

OVERWRITE_ORIGINAL_CLEAN_JSON = True

patched_items = list(clean_items)

patched_manual_item = dict(manual_repair_item)
patched_manual_item["final_clean_source_file"] = MANUAL_REPAIR_ITEM_PATH.name
patched_manual_item["final_clean_source_kind"] = "manual_repair"

patched_items[MANUAL_INPUT_INDEX] = patched_manual_item

# Save a separate manually repaired clean file.
atomic_json_dump(patched_items, CLEAN_MANUAL_REPAIRED_PATH)

print("Saved manually repaired clean JSON:", CLEAN_MANUAL_REPAIRED_PATH)

if OVERWRITE_ORIGINAL_CLEAN_JSON:
    atomic_json_dump(patched_items, CLEAN_OUTPUT_PATH)
    print("Overwrote original clean JSON after backup:", CLEAN_OUTPUT_PATH)
else:
    print("Original clean JSON was not overwritten.")

Saved manually repaired clean JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.manual_repaired.json
Overwrote original clean JSON after backup: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.json


In [6]:
# 6) Re-audit the repaired clean JSON

AUDIT_TARGET_PATH = CLEAN_OUTPUT_PATH if OVERWRITE_ORIGINAL_CLEAN_JSON else CLEAN_MANUAL_REPAIRED_PATH

with open(AUDIT_TARGET_PATH, "r", encoding="utf-8") as f:
    final_items = json.load(f)

assert isinstance(final_items, list), "Final clean JSON must be a list."
assert len(final_items) == TOTAL_CHUNKS, f"Expected {TOTAL_CHUNKS} items, got {len(final_items)}."

missing_indices = []
unusable_indices = []
chunk_id_mismatches = []
failure_reasons_by_input_index = {}

for idx in range(TOTAL_CHUNKS):
    item = final_items[idx]

    if not isinstance(item, dict):
        unusable_indices.append(idx)
        failure_reasons_by_input_index[str(idx)] = ["item_not_dict"]
        continue

    if item.get("chunk_id") != expected_chunk_id(idx):
        chunk_id_mismatches.append({
            "input_index": idx,
            "expected_chunk_id": expected_chunk_id(idx),
            "got_chunk_id": item.get("chunk_id"),
        })

    if not is_usable_success(item, idx):
        unusable_indices.append(idx)
        failure_reasons_by_input_index[str(idx)] = failure_reasons_hotpot_style(item, idx)

# Because final_items is a list with one item per index, missing is only possible if length was wrong.
if len(final_items) != TOTAL_CHUNKS:
    missing_indices = sorted(set(range(TOTAL_CHUNKS)) - set(range(len(final_items))))

chosen_sources = Counter(
    item.get("final_clean_source_kind", "unknown")
    for item in final_items
    if isinstance(item, dict)
)

manual_item_after_patch = final_items[MANUAL_INPUT_INDEX]
manual_item_strict_errors_after_patch = validate_manual_kg_payload({
    "entities": manual_item_after_patch.get("entities"),
    "relations": manual_item_after_patch.get("relations"),
    "facts": manual_item_after_patch.get("facts"),
})

final_audit = {
    "dataset": "2wikimultihopqa",
    "audit_target_path": str(AUDIT_TARGET_PATH),
    "manual_repair_item_path": str(MANUAL_REPAIR_ITEM_PATH),
    "total_chunks": TOTAL_CHUNKS,
    "num_items": len(final_items),
    "num_missing_indices": len(missing_indices),
    "num_unusable_indices": len(unusable_indices),
    "num_chunk_id_mismatches": len(chunk_id_mismatches),
    "missing_input_index_ranges_0_based": summarize_ranges(missing_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in missing_indices]),
    "unusable_input_index_ranges_0_based": summarize_ranges(unusable_indices),
    "unusable_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in unusable_indices]),
    "chunk_id_mismatches": chunk_id_mismatches,
    "failure_reasons_by_input_index": failure_reasons_by_input_index,
    "manual_repaired_input_index_0_based": MANUAL_INPUT_INDEX,
    "manual_repaired_chunk_number_1_based": MANUAL_CHUNK_NUMBER,
    "manual_repaired_chunk_id": manual_chunk["Chunk_id"],
    "manual_item_strict_errors_after_patch": manual_item_strict_errors_after_patch,
    "chosen_sources": dict(chosen_sources),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

# Save both a manual-repaired audit and optionally overwrite the standard clean audit.
atomic_json_dump(final_audit, CLEAN_MANUAL_REPAIRED_AUDIT_PATH)

if OVERWRITE_ORIGINAL_CLEAN_JSON:
    atomic_json_dump(final_audit, CLEAN_AUDIT_PATH)

print("Saved manual-repaired audit:", CLEAN_MANUAL_REPAIRED_AUDIT_PATH)

if OVERWRITE_ORIGINAL_CLEAN_JSON:
    print("Overwrote standard clean audit:", CLEAN_AUDIT_PATH)

print("Final items:", len(final_items), "/", TOTAL_CHUNKS)
print("Missing:", len(missing_indices))
print("Unusable:", len(unusable_indices))
print("Unusable chunk numbers:", summarize_ranges([i + 1 for i in unusable_indices]))
print("Chunk id mismatches:", len(chunk_id_mismatches))
print("Manual item strict errors after patch:", manual_item_strict_errors_after_patch)

if missing_indices:
    raise RuntimeError("Some chunks are missing after manual repair.")

if chunk_id_mismatches:
    raise RuntimeError("Some chunk_id values do not match their input_index after manual repair.")

if unusable_indices:
    raise RuntimeError("Some chunks are still unusable after manual repair.")

if manual_item_strict_errors_after_patch:
    raise RuntimeError("Manual item strict KG validation failed after patch.")

print("Manual repair audit passed. All chunks are usable.")

Saved manual-repaired audit: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.manual_repaired_audit.json
Overwrote standard clean audit: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean_audit.json
Final items: 12685 / 12685
Missing: 0
Unusable: 0
Unusable chunk numbers: []
Chunk id mismatches: 0
Manual item strict errors after patch: []
Manual repair audit passed. All chunks are usable.


In [7]:
# 7) Update unresolved chunks file and print final confirmation

# If audit passed, unresolved list should be empty.
unresolved_details = []

atomic_json_dump(unresolved_details, UNRESOLVED_CHUNKS_PATH)

print("Saved unresolved chunks JSON:", UNRESOLVED_CHUNKS_PATH)
print("Number of unresolved chunks:", len(unresolved_details))

print("\nManual repaired item at index:")
print("input_index_0_based:", MANUAL_INPUT_INDEX)
print("chunk_number_1_based:", MANUAL_CHUNK_NUMBER)
print("chunk_id:", final_items[MANUAL_INPUT_INDEX]["chunk_id"])
print("title:", final_items[MANUAL_INPUT_INDEX]["title"])
print("error:", final_items[MANUAL_INPUT_INDEX]["error"])
print("finish_reason:", final_items[MANUAL_INPUT_INDEX]["finish_reason"])
print("final_clean_source_file:", final_items[MANUAL_INPUT_INDEX].get("final_clean_source_file"))
print("final_clean_source_kind:", final_items[MANUAL_INPUT_INDEX].get("final_clean_source_kind"))

print("\nPaths:")
print("Final clean JSON:", CLEAN_OUTPUT_PATH if OVERWRITE_ORIGINAL_CLEAN_JSON else CLEAN_MANUAL_REPAIRED_PATH)
print("Final clean audit:", CLEAN_AUDIT_PATH if OVERWRITE_ORIGINAL_CLEAN_JSON else CLEAN_MANUAL_REPAIRED_AUDIT_PATH)
print("Manual repair item:", MANUAL_REPAIR_ITEM_PATH)
print("Manual repair meta:", MANUAL_REPAIR_META_PATH)
print("Clean backup:", CLEAN_BACKUP_PATH)

print("\nManual repaired KG payload:")
print(json.dumps({
    "entities": final_items[MANUAL_INPUT_INDEX]["entities"],
    "relations": final_items[MANUAL_INPUT_INDEX]["relations"],
    "facts": final_items[MANUAL_INPUT_INDEX]["facts"],
}, ensure_ascii=False, indent=2))

print("\nDone. The final clean JSON has no unresolved chunks.")

Saved unresolved chunks JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.json
Number of unresolved chunks: 0

Manual repaired item at index:
input_index_0_based: 10636
chunk_number_1_based: 10637
chunk_id: 2wikimultihopqa_chunk_00010637
title: Youssef Chahine
error: None
finish_reason: manual_repair
final_clean_source_file: 2wikimultihopqa_kg_manual_repair_chunk_00010637.json
final_clean_source_kind: manual_repair

Paths:
Final clean JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.json
Final clean audit: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean_audit.json
Manual repair item: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_manual_repair_chunk_00010637.json
Manual repair meta: /content/drive/MyDrive/final_project/ide